<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#@title 1. Install dependencies
!pip install -q "torchao>=0.16.0"
!pip install -q diffusers transformers accelerate peft safetensors pillow opencv-python-headless

from IPython.display import clear_output
clear_output()
print("✅ Installed")

✅ Installed


In [5]:
#@title 2. Load Qwen-Image + BFS Head Swap LoRA
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import load_image
from huggingface_hub import hf_hub_download
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print("Loading base model (this takes 1-2 minutes)...")

# Remove device_map so that enable_model_cpu_offload works properly
pipe = DiffusionPipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16
)

print("Loading BFS Head Swap LoRA...")
pipe.load_lora_weights(
    "Alissonerdx/BFS-Best-Face-Swap",
    weight_name="bfs_head_v1.1_qwen_2.1.safetensors"
)

# Apply efficient offloading and optimizations
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

print("✅ Model + LoRA loaded")

Loading base model (this takes 1-2 minutes)...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading BFS Head Swap LoRA...


Loading adapter weights from state_dict led to unexpected keys found in the model: transformer_blocks.0.img_mlp.gate_up.lora_A.weight, transformer_blocks.0.img_mlp.gate_up.lora_B.weight, transformer_blocks.1.img_mlp.gate_up.lora_A.weight, transformer_blocks.1.img_mlp.gate_up.lora_B.weight, transformer_blocks.2.img_mlp.gate_up.lora_A.weight, transformer_blocks.2.img_mlp.gate_up.lora_B.weight, transformer_blocks.3.img_mlp.gate_up.lora_A.weight, transformer_blocks.3.img_mlp.gate_up.lora_B.weight, transformer_blocks.4.img_mlp.gate_up.lora_A.weight, transformer_blocks.4.img_mlp.gate_up.lora_B.weight, transformer_blocks.5.img_mlp.gate_up.lora_A.weight, transformer_blocks.5.img_mlp.gate_up.lora_B.weight, transformer_blocks.6.img_mlp.gate_up.lora_A.weight, transformer_blocks.6.img_mlp.gate_up.lora_B.weight, transformer_blocks.7.img_mlp.gate_up.lora_A.weight, transformer_blocks.7.img_mlp.gate_up.lora_B.weight, transformer_blocks.8.img_mlp.gate_up.lora_A.weight, transformer_blocks.8.img_mlp.gate

✅ Model + LoRA loaded


In [22]:
import gc
import torch

# Free model from notebook kernel memory so the backend server has full VRAM access
if 'pipe' in globals():
    del pipe

gc.collect()
torch.cuda.empty_cache()
print("✅ Notebook kernel GPU memory cleared. VRAM is now fully available for the background server!")

✅ Notebook kernel GPU memory cleared. VRAM is now fully available for the background server!


In [23]:
%%writefile /content/headswap_server.py
import os
import sys
import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from diffusers import DiffusionPipeline
import base64
from io import BytesIO
from PIL import Image
import gc

app = FastAPI()

# Configure CORS middleware to allow requests from your frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allows all origins, you can restrict this to your domain if needed
    allow_credentials=True,
    allow_methods=["*"],  # Allows all methods (POST, OPTIONS, etc.)
    allow_headers=["*"],  # Allows all headers
)

pipe = None

class HeadSwapRequest(BaseModel):
    body: str
    head: str
    prompt: str = ""

def dataurl_to_pil(dataurl: str) -> Image.Image:
    if "," in dataurl:
        dataurl = dataurl.split(",")[1]
    image_data = base64.b64decode(dataurl)
    return Image.open(BytesIO(image_data)).convert("RGB")

def pil_to_dataurl(img: Image.Image) -> str:
    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{img_str}"

@app.on_event("startup")
def load_model():
    global pipe
    print("Loading Qwen-Image 2.1 + BFS Head Swap LoRA...", flush=True)
    torch.cuda.empty_cache()
    gc.collect()

    pipe = DiffusionPipeline.from_pretrained(
        "Qwen/Qwen-Image-2.1",
        torch_dtype=torch.bfloat16
    )
    pipe.load_lora_weights(
        "Alissonerdx/BFS-Best-Face-Swap",
        weight_name="bfs_head_v1.1_qwen_2.1.safetensors"
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing()
    print("Model + LoRA loaded", flush=True)

@app.post("/headswap")
def headswap(req: HeadSwapRequest):
    try:
        body_img = dataurl_to_pil(req.body)
        head_img = dataurl_to_pil(req.head)

        prompt = req.prompt.strip() or (
            "head_swap: start with <image1> as the base image, keeping its lighting, environment, background, clothing, skin texture, and any fluids (sweat, tears, saliva, cum, spit, wetness) exactly as they appear on <image1>. remove the head from <image1> completely and replace it with the head from <image2>, strictly preserving the hair, eye color, nose structure, and facial identity from <image2>. copy the head rotation, eye direction, and micro expressions from <image1>. keep all fluids, wetness, and liquid details from the original body image. high quality, sharp details, 4k"
        )

        print(f"Running head swap... body={body_img.size}, head={head_img.size}", flush=True)

        # Correct model invocation without unsupported parameters
        result = pipe(
            prompt=prompt,
            image=[body_img, head_img],
            num_inference_steps=12,
            height=1024,
            width=1024
        ).images[0]

        print("Done.", flush=True)
        return {"image": pil_to_dataurl(result)}

    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(error_msg, flush=True)
        return {"error": str(e)}

if __name__ == "__main__":
    import uvicorn
    print("Server listening...", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000)

Overwriting /content/headswap_server.py


In [ ]:
#@title 4. Start server + Cloudflare tunnel
import os, re, sys, time, subprocess, threading

# Install server dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"], check=True)

SERVER_SCRIPT = "/content/headswap_server.py"
SERVER_LOG = "/content/headswap_server.log"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Kill any previous server / tunnel
subprocess.run(["pkill", "-f", "headswap_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Download cloudflared if needed
if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run([
        "wget", "-q", "-O", CLOUDFLARED,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    os.chmod(CLOUDFLARED, 0o755)

# Start the FastAPI server
print("Starting Head Swap server...")
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        [sys.executable, SERVER_SCRIPT],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

# Wait until the model is loaded
print("Waiting for model to load (this can take 1–2 minutes)...")
for _ in range(180):
    time.sleep(2)
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
            if "Model + LoRA loaded" in content or "Server listening" in content:
                print("✅ Model loaded")
                break
else:
    print("⚠️ Model took too long. Check the log below.")
    !tail -30 /content/headswap_server.log

# Start Cloudflare tunnel
print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
        break

if TUNNEL_URL:
    print("\n" + "="*60)
    print(f"READY: {TUNNEL_URL}")
    print("="*60)
    print("\n→ Copy the URL above and paste it into the Head Swap tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")

Starting Head Swap server...
Waiting for model to load (this can take 1–2 minutes)...


In [20]:
!tail -50 /content/headswap_server.log

        width=1024
        ^^^^^^^^^^
    ).images[0]
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/diffusers/pipelines/qwenimage21/pipeline_qwenimage21.py", line 784, in __call__
    noise_pred = self.transformer(
                 ~~~~~~~~~~~~~~~~^
        hidden_states=latent_model_input,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
        return_dict=False,
        ^^^^^^^^^^^^^^^^^^
    )[0]
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1794, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/accelerate/hooks.py", line 192, in new_f